# 06 · Runtime and the decode loop

This notebook covers how the compiled artifact is loaded and called, and how its decode loop lines up
with HF's:
- how the runner loads the artifact (VM, packed params, static-shape dispatch);
- the attention callback, observed while it runs;
- zero-copy DLPack handoff between TVM and PyTorch;
- functional caches, step by step;
- one generation traced call by call, side by side with the HF trace from notebook 01.

**Needs:** CUDA, `artifacts/fast_dllm_relax_bf16`, and the checkpoint (for the HF side-by-side). A
few minutes. Peak GPU memory is about 8 GB.

In [1]:
from _helpers import *
bootstrap()

repo root: fast-dllm-relax/  (cwd set)  tvm 0.24.dev0  torch 2.10.0+cu128


PosixPath('<repo>')

In [2]:
import time
import numpy as np
import torch
import tvm
from fast_dllm_relax import runtime as rt   # importing it registers the attention callback

runner = load_relax("artifacts/fast_dllm_relax_bf16", max_new_tokens=64)
vm, P, dev = runner.vm, runner.params, runner.dev
print(f"params: {len(P)} tensors, first {P[0].shape} {P[0].dtype}  (from the npz, in param_names.txt order)")
print("dispatch:", rt.selected_fn_names(vm, block_size=32, small_block_size=8))

params: 198 tensors, first (151936, 1536) bfloat16  (from the npz, in param_names.txt order)
dispatch: {'cached_forward_fn': 'forward_cached_block', 'append_forward_fn': 'forward_cached', 'logits_fn': 'get_logits', 'partial_logits_fn': 'get_logits'}


Loading (`FastDLLMRelaxRunner.load` in `runners.py`) takes four steps:
1. `tvm.runtime.load_module(.so)` → `relax.VirtualMachine(ex, dev)`.
2. Check that the cache-aware globals exist.
3. Read the npz into a list of TVM tensors in export order. bf16 is stored as uint16 bits and
   reinterpreted on load.
4. Fetch the `vm.builtin.reshape` builtin, used to add the batch dim after `embed`.

CUDA artifacts have no static-shape siblings, so dispatch uses the dynamic functions. Metal
artifacts have `_b32`/`_m*` variants (notebook 08).

## The attention callback, observed

`forward_cached*` and the partial functions call the Python function registered as
`fast_dllm.flash_attn_bf16` once per layer. The VM resolves extern functions when it is
**created**, so to observe the callback, register a counting wrapper and then make a new VM from the
same executable:

In [3]:
orig_cb = tvm.get_global_func("fast_dllm.flash_attn_bf16")
calls = []
def counting_cb(q, k, v):
    calls.append({"q (BSNH)": tuple(q.shape), "k/v (BSNH)": tuple(k.shape), "dtype": str(q.dtype)})
    return orig_cb(q, k, v)
tvm.register_global_func("fast_dllm.flash_attn_bf16", counting_cb, override=True)
vm_obs = tvm.relax.VirtualMachine(runner.loaded_ex, dev)
tvm.register_global_func("fast_dllm.flash_attn_bf16", orig_cb, override=True)   # later VMs get the original

ids = chat_ids(runner.tokenizer, 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?')
L = len(ids)
a0 = L // 32 * 32
pc = rt.build_prefix_cache_from_ids(vm_obs, P, dev, runner.reshape, ids[:a0], num_attention_heads=12, block_size=32)
block = ids[a0:] + [runner.mask_id] * (32 - L % 32)
_ = rt.score_active_block_refine(vm_obs, P, dev, runner.reshape, block, pc, block_size=32, small_block_size=8)
print(f"build_prefix_cache: in-graph attention (no callback).  one refinement step: {len(calls)} callbacks")
print_table(calls[:3]); print("...")

build_prefix_cache: in-graph attention (no callback).  one refinement step: 28 callbacks
q (BSNH)          k/v (BSNH)       dtype   
----------------  ---------------  --------
(1, 32, 12, 128)  (1, 64, 2, 128)  bfloat16
(1, 32, 12, 128)  (1, 64, 2, 128)  bfloat16
(1, 32, 12, 128)  (1, 64, 2, 128)  bfloat16
...


- `build_prefix_cache` computes its attention in-graph, using the explicit fp32 path with the
  block-causal bias.
- Each cached step makes one callback per layer (28). The query is the 32-token block; keys and
  values are prefix + block.
- GQA is handled natively: 2 K/V heads against 12 query heads, with no repeat.

## DLPack: crossing between TVM and PyTorch for free

`torch.from_dlpack(tvm_tensor)` wraps the same GPU memory and copies nothing. Its cost does not
depend on size, unlike a host copy:

In [4]:
lg = vm["get_logits"](vm["forward_cached_block"](
        runner.reshape(vm["embed"](tvm.runtime.tensor(np.asarray(block, "int32"), device=dev), P),
                       tvm.runtime.ShapeTuple([1, 32, 1536])),
        pc.cached_k, pc.cached_v, tvm.runtime.tensor(np.asarray(pc.prefix_len, "int32"), device=dev), P)[0], P)
dev.sync()
t0 = time.perf_counter(); view = torch.from_dlpack(lg); t1 = time.perf_counter()
host = lg.numpy(); t2 = time.perf_counter()
print(f"logits {tuple(view.shape)} fp32 = {view.numel() * 4 / 1e6:.1f} MB")
print(f"torch.from_dlpack: {(t1 - t0) * 1e6:.0f} us (view, device={view.device})   .numpy(): {(t2 - t1) * 1e3:.2f} ms (device->host copy)")

logits (1, 32, 151936) fp32 = 19.4 MB
torch.from_dlpack: 74 us (view, device=cuda:0)   .numpy(): 3.49 ms (device->host copy)


## Functional caches, step by step

Each compiled call returns **new** cache tensors. Old ones are freed when nothing references them
any more. Python only holds handles (`PrefixCache`, `BlockCache`).

In [5]:
def shape(t): return tuple(int(d) for d in t.shape)
rows = [{"step": "build_prefix_cache", "prefix cache": shape(pc.cached_k), "returned K": shape(pc.cached_k)}]
ids_r, probs, bc = rt.score_active_block_refine(vm, P, dev, runner.reshape, block, pc, block_size=32, small_block_size=8)
rows.append({"step": "refine (forward_cached_block)", "prefix cache": shape(pc.cached_k), "returned K": shape(bc.block_k)})
_, _, nxt, pc2 = rt.score_active_block_cached_fast(vm, P, dev, runner.reshape, block, pc)
rows.append({"step": "append (forward_cached)", "prefix cache": shape(pc2.cached_k), "returned K": shape(pc2.cached_k)})
print_table(rows)

step                           prefix cache         returned K         
-----------------------------  -------------------  -------------------
build_prefix_cache             (28, 1, 2, 32, 128)  (28, 1, 2, 32, 128)
refine (forward_cached_block)  (28, 1, 2, 32, 128)  (28, 1, 2, 32, 128)
append (forward_cached)        (28, 1, 2, 64, 128)  (28, 1, 2, 64, 128)


- **Refinement** steps never change the prefix cache, so `forward_cached_block` returns only the
  block's K/V: `[28, 1, 2, 32, 128]`.
- **Append** returns the whole extended cache, which is the one full O(prefix) copy per block.

## One generation, call by call

`relax_traced_generate` (in `_helpers.py`) re-implements `runtime.relax_greedy_generate`
(`use_block_cache=False`) on top of the runtime's own step functions, and records each call. First,
check that it is faithful:

In [6]:
cont_traced, rx_events = relax_traced_generate(runner, ids, max_new_tokens=64)
cont_ref, _stats = rt.relax_greedy_generate(
    vm, P, dev, runner.reshape, runner.tokenizer, ids, num_attention_heads=12, max_new_tokens=64,
    block_size=32, small_block_size=8, threshold=0.9, temperature=0.0, top_p=0.95,
    use_block_cache=False, mask_id=runner.mask_id, stop_token=runner.stop_token)
print("trace reproduces relax_greedy_generate:", cont_traced == cont_ref)
events_table(rx_events, runner.tokenizer)

trace reproduces relax_greedy_generate: True
step  kind     fn                    tokens_in  cache_len  sub_block  rows                      conf                                                            text                                  
----  -------  --------------------  ---------  ---------  ---------  ------------------------  --------------------------------------------------------------  --------------------------------------
0     prefill  build_prefix_cache    32         32                                                                                                                                                    
1     full     forward_cached_block  32         32         3          [2, 3, 4, 5, 6, 7]        [0.9948, 0.9922, 0.9636, 0.9685, 0.9543, 0.9533]                'To determine the total number of'    
2     append   forward_cached        32         64                                                                                                             

Now the same prompt through HF (as in notebook 01), and the two traces side by side:

In [7]:
tok, hf = load_hf()
x = torch.tensor([ids], device="cuda")
hf_ids, hf_events = hf_traced_generate(hf, x, max_new_tokens=64)
del hf; torch.cuda.empty_cache()

def norm(evts):
    return [e for e in evts if e["kind"] == "full"]
rows = []
for i, (h, r) in enumerate(zip(norm(hf_events), norm(rx_events))):
    rows.append({"full step": i, "sub-block": h["sub_block"], "HF rows": h["unmasked_rows"], "Relax rows": r["unmasked_rows"],
                 "same tokens": h["tokens"] == r["tokens"] and h["unmasked_rows"] == r["unmasked_rows"],
                 "HF conf": h["conf"], "Relax conf": r["conf"]})
print_table(rows[:30])
hf_cont = hf_ids[L:]
n = min(len(hf_cont), len(cont_ref))
first_diff = next((i for i in range(n) if hf_cont[i] != cont_ref[i]), None)
print(f"\ncontinuations identical for the first {n if first_diff is None else first_diff} of {n} tokens")

full step  sub-block  HF rows                   Relax rows                same tokens  HF conf                                                     Relax conf                                                    
---------  ---------  ------------------------  ------------------------  -----------  ----------------------------------------------------------  --------------------------------------------------------------
0          3          [2, 3, 4, 5, 6, 7]        [2, 3, 4, 5, 6, 7]        True         [0.9961, 0.9922, 0.9648, 0.9727, 0.9531, 0.9531]            [0.9948, 0.9922, 0.9636, 0.9685, 0.9543, 0.9533]              
1          0          [1, 2, 3, 4]              [1, 2, 3, 4]              True         [0.9961, 0.9961, 1.0, 0.9805]                               [0.9979, 0.9968, 0.9989, 0.9787]                              
2          0          [5, 7]                    [5, 7]                    True         [0.9297, 0.9453]                                            [0.9229, 0.95

Each row is one full forward pass. Matching rows mean both backends unmasked the same positions with
the same tokens. The confidences differ in the low digits (notebook 03). A row stops matching when a
confidence lands on the other side of 0.9, or when an argmax flips. After that, the two sequences
are different inputs and are no longer comparable step for step.

## Takeaways
- The runtime is a **Python loop over compiled steps**. TVM runs the GEMMs and fused kernels, PyTorch
  runs cached attention (the callback) and the softmax/argmax, and numpy/Python decide what to unmask.
- Handoffs between TVM and PyTorch are zero-copy views. The cost of the callback is the host round
  trip per layer, not data movement (measured in notebook 07).
- The two decode loops are structurally identical: prefill, then full steps per sub-block, then append.
- Next: **07 · Profiling**.